# 0️⃣2️⃣ - Managing your devices
![Persona](https://img.shields.io/badge/%F0%9F%91%A4%20Persona-%F0%9F%9B%A0%EF%B8%8F%20Platform%20%2F%20IT%20Administrator-lightgrey) ![Difficulty](https://img.shields.io/badge/%F0%9F%9A%A6%20Difficulty-Beginner-green) ![RADKit version](https://img.shields.io/badge/RADKit-1.9.6-blue?logo=cisco&logoColor=white) ![Python version](https://img.shields.io/badge/Python-3.12%2B-yellow?logo=python&logoColor=white)

> In this playbook we will explore how to create, list, update, and delete devices.

---

### 📋 What you'll learn

| # | Topic |
|---|-------|
| 1 | How to connect to a RADKit service using the `ControlAPI` context manager |
| 2 | How to list all devices and retrieve a specific one by name or UUID |
| 3 | How to create a new device with terminal credentials and a `DeviceType` enum |
| 4 | How to update the attributes of an existing device |
| 5 | How to delete a device |

### 🛠️ Before You Begin

If you haven't set up your environment yet, follow the instructions in [SETUP.md](../SETUP.md) before running any cells.

---

### 🤖 The `ControlAPI` Object

The `ControlAPI` object is your main entry point for administering a RADKit service programmatically. It acts as an alternative to the WebUI, giving you full control over service components: devices, remote users, labels, admins, and the service itself.

Used as a context manager, it authenticates as a service superadmin and returns a live handle through which all subsequent API calls are made. All examples in this notebook use this pattern.

---

## 📋 Method 1: Retrieve Devices

**Best for:** Auditing the current device inventory, pre-flight checks before automation runs, or pulling the exact attribute state of a specific device.

**How it works:**
1. Connect to the service with `ControlAPI.create()`.
2. Call `list_devices()` to get all configured devices. Pass `exclude_metadata=True` to strip metadata from the response.
3. For a specific device, call `get_device(device_uuid)` using the device's UUID.
4. Each result is a Pydantic model instance containing all device attributes, including terminal, NETCONF, SNMP, and HTTP settings.

**What you need:**
- The RADKit server address and superadmin password
- The name or UUID of the device you want to inspect

### 1.1 List all devices and inspect a specific one

In [ ]:
from getpass import getpass
from radkit_service.control_api import ControlAPI

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:
    
    # Fetch the UUID of a device by its name
    device_uuid = None
    for device in service.list_devices().result:
        if device.name == 'p0-2e':
            device_uuid = device.uuid
        
    # Information about a specific device
    if device_uuid is not None:
        my_device = service.get_device(device_uuid).root.result[0]
        
        print(f"🆔 UUID:               {my_device.uuid}")
        print(f"📛 Name:               {my_device.name}")
        print(f"🌐 Host:               {my_device.host}")
        print(f"🖥️ Device Type:        {my_device.deviceType}")
        print(f"🔀 Jumphost Ref:       {my_device.jumphostRef}")
        print(f"✅ Enabled:            {my_device.enabled}")
        print(f"\n---------------\n🔌 Forwarded TCP Ports:{my_device.forwardedTcpPorts!r}")
        print(f"🏷️  Labels:             {my_device.labels}")
        print(f"📝 Description:        {my_device.description!r}")
        print(f"🔑 Source Key:         {my_device.sourceKey}")
        print(f"🔗 Source Dev UUID:    {my_device.sourceDevUuid}")
        print(f"\n---------------\n💻 Terminal:")
        print(f"   🔗 Connection Method:      {my_device.terminal.connectionMethod}")
        print(f"   🔢 Port:                   {my_device.terminal.port}")
        print(f"   👤 Username:               {'*' * len(my_device.terminal.username)}")
        print(f"   🔓 Enable Set:             {my_device.terminal.enableSet}")
        print(f"   ⚠️  Use Insecure Algorithms:{my_device.terminal.useInsecureAlgorithms}")
        print(f"   🏂 Jumphost:               {my_device.terminal.jumphost}")
        print(f"   🚇 Use Tunneling:          {my_device.terminal.useTunnelingIfJumphost}")
        print(f"   📦 Provisioning Variant:   {my_device.terminal.provisioningVariant}")
        print(f"   🛠️ Capabilities:")
        for cap in my_device.terminal.capabilities:
            print(f"      • {cap}")
        try:
            print(f"\n---------------\n🔧 NETCONF:")
            print(f"   🔢 Port:                   {my_device.netconf.port}")
            print(f"   👤 Username:               {my_device.netconf.username}")
            print(f"   ⚠️ Use Insecure Algorithms:{my_device.netconf.useInsecureAlgorithms}")
        except AttributeError:
            pass
        try:
            print(f"\n---------------\n📡 SNMP:")
            print(f"   📋 Version:                {my_device.snmp.version}")
            print(f"   🔢 Port:                   {my_device.snmp.port}")
        except AttributeError:
            pass
        print(f"\n---------------\n🕸️ Swagger:            {my_device.swagger}")
        print(f"🌍 HTTP:               {my_device.http}")
        print(f"📄 Device Template Ref:{my_device.deviceTemplateRef}")
        print(f"ℹ️ Metadata:           {my_device.metaData}")


🆔 UUID:               86b72618-c05e-49fc-b3d3-adc50d230c81
📛 Name:               p0-2e
🌐 Host:               10.48.172.59
🖥️ Device Type:        DeviceType.IOS_XE
🔀 Jumphost Ref:       None
✅ Enabled:            True

---------------
🔌 Forwarded TCP Ports:''
🏷️  Labels:             {4}
📝 Description:        ''
🔑 Source Key:         None
🔗 Source Dev UUID:    None

---------------
💻 Terminal:
   🔗 Connection Method:      ConnectionMethod.SSH
   🔢 Port:                   22
   👤 Username:               ********
   🔓 Enable Set:             True
   ⚠️  Use Insecure Algorithms:False
   🏂 Jumphost:               False
   🚇 Use Tunneling:          True
   📦 Provisioning Variant:   ProvisioningVariant.DEFAULT
   🛠️ Capabilities:
      • TerminalCapabilities.UPLOAD
      • TerminalCapabilities.INTERACTIVE
      • TerminalCapabilities.DOWNLOAD
      • TerminalCapabilities.EXEC

---------------
🔧 NETCONF:
   🔢 Port:                   830
   👤 Username:               netconfdemo
   ⚠️ Use Insecur

---

## ➕ Method 2: Create a Device

**Best for:** Automating device provisioning: bootstrapping new inventory entries, scripting initial configuration as part of a CI/CD pipeline, or seeding a service with devices from an external source of truth.

**How it works:**
1. Create a `NewTerminal` instance with the device's SSH credentials if terminal access is required.
2. Create a `NewDevice` instance with the device name, host, `DeviceType` enum, and any optional fields.
3. Call `create_device()` to register the device on the service.
4. Confirm creation by fetching the new device with `get_device()`.

**What you need:**
- The RADKit server address and superadmin password
- The target device's hostname or IP address, device type, and credentials

> **About device names:** The name must comply with RADKit formatting rules. Use `to_canonical_name()` to transform any arbitrary string into a compliant name.

> **About `DeviceType`:** Use the `DeviceType` enum (e.g. `DeviceType.IOS_XR`) instead of raw strings to avoid typos and get IDE completion.

### 2.1 Create a device with terminal access

In [ ]:
from getpass import getpass
from radkit_common.utils.formatting import to_canonical_name
from radkit_service.control_api import ControlAPI, NewTerminal, NewDevice, DeviceType

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:
    
    new_terminal = NewTerminal(
            username="admin",
            password="Cisco123",
    )
    
    hostname = "dummy_iosxr_demo"
    # Creation of a new device
    service.create_device(
        NewDevice(
            name = to_canonical_name(hostname),
            host = "192.168.5.2",
            deviceType = DeviceType.IOS_XR,
            enabled=False,
            description="This is a dummy device created via the Control API",
            terminal=new_terminal
        ))
    
    # Fetch the UUID of a device by its name
    device_uuid = None
    for device in service.list_devices().result:
        if device.name == to_canonical_name(hostname):
            device_uuid = device.uuid
        
    # Information about a specific device
    if device_uuid is not None:
        my_device = service.get_device(device_uuid).root.result[0]
        
        print(f"🆔 UUID:               {my_device.uuid}")
        print(f"📛 Name:               {my_device.name}")
        print(f"🌐 Host:               {my_device.host}")
        print(f"🖥️ Device Type:        {my_device.deviceType}")
        print(f"🔀 Jumphost Ref:       {my_device.jumphostRef}")
        print(f"✅ Enabled:            {my_device.enabled}")
        print(f"\n---------------\n🔌 Forwarded TCP Ports:{my_device.forwardedTcpPorts!r}")
        print(f"🏷️  Labels:             {my_device.labels}")
        print(f"📝 Description:        {my_device.description!r}")
        print(f"🔑 Source Key:         {my_device.sourceKey}")
        print(f"🔗 Source Dev UUID:    {my_device.sourceDevUuid}")
        print(f"\n---------------\n💻 Terminal:")
        print(f"   🔗 Connection Method:      {my_device.terminal.connectionMethod}")
        print(f"   🔢 Port:                   {my_device.terminal.port}")
        print(f"   👤 Username:               {'*' * len(my_device.terminal.username)}")
        print(f"   🔓 Enable Set:             {my_device.terminal.enableSet}")
        print(f"   ⚠️  Use Insecure Algorithms:{my_device.terminal.useInsecureAlgorithms}")
        print(f"   🏂 Jumphost:               {my_device.terminal.jumphost}")
        print(f"   🚇 Use Tunneling:          {my_device.terminal.useTunnelingIfJumphost}")
        print(f"   📦 Provisioning Variant:   {my_device.terminal.provisioningVariant}")
        print(f"   🛠️ Capabilities:")
        for cap in my_device.terminal.capabilities:
            print(f"      • {cap}")
        try:
            print(f"\n---------------\n🔧 NETCONF:")
            print(f"   🔢 Port:                   {my_device.netconf.port}")
            print(f"   👤 Username:               {my_device.netconf.username}")
            print(f"   ⚠️ Use Insecure Algorithms:{my_device.netconf.useInsecureAlgorithms}")
        except AttributeError:
            pass
        try:
            print(f"\n---------------\n📡 SNMP:")
            print(f"   📋 Version:                {my_device.snmp.version}")
            print(f"   🔢 Port:                   {my_device.snmp.port}")
        except AttributeError:
            pass
        print(f"\n---------------\n🕸️ Swagger:            {my_device.swagger}")
        print(f"🌍 HTTP:               {my_device.http}")
        print(f"📄 Device Template Ref:{my_device.deviceTemplateRef}")
        print(f"ℹ️ Metadata:           {my_device.metaData}")
    else:
        print("Device not found.")


🆔 UUID:               da5d338c-8ff8-4a69-98d6-6d897cfd3bf3
📛 Name:               dummy-iosxr-demo
🌐 Host:               192.168.5.2
🖥️ Device Type:        DeviceType.IOS_XR
🔀 Jumphost Ref:       None
✅ Enabled:            False

---------------
🔌 Forwarded TCP Ports:''
🏷️  Labels:             set()
📝 Description:        'This is a dummy device created via the Control API'
🔑 Source Key:         None
🔗 Source Dev UUID:    None

---------------
💻 Terminal:
   🔗 Connection Method:      ConnectionMethod.SSH
   🔢 Port:                   22
   👤 Username:               *****
   🔓 Enable Set:             False
   ⚠️  Use Insecure Algorithms:False
   🏂 Jumphost:               False
   🚇 Use Tunneling:          True
   📦 Provisioning Variant:   ProvisioningVariant.DEFAULT
   🛠️ Capabilities:
      • TerminalCapabilities.UPLOAD
      • TerminalCapabilities.INTERACTIVE
      • TerminalCapabilities.DOWNLOAD
      • TerminalCapabilities.EXEC

---------------
🔧 NETCONF:

---------------
📡 SNMP:

----

---

## ✏️ Method 3: Update a Device

**Best for:** Changing device attributes after provisioning: enabling or disabling a device, updating its IP address, or modifying description and labels without recreating the entry.

**How it works:**
1. Locate the target device's UUID by iterating `list_devices()` or from a previous step.
2. Create an `UpdateDevice` instance with the UUID and only the fields you want to change.
3. Call `update_device()` to apply the changes.
4. Confirm by fetching the device again with `get_device()`.

> **Note:** Fields you omit in `UpdateDevice` retain their current values.

In [2]:
from getpass import getpass
from radkit_common.utils.formatting import to_canonical_name
from radkit_service.control_api import ControlAPI, UpdateDevice

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:
        
    # Fetch the UUID of a device by its name
    device_uuid = None
    for device in service.list_devices().result:
        if device.name == to_canonical_name("dummy_iosxr_demo"):
            device_uuid = device.uuid
        
    if device_uuid is not None:
        
        # Updating the device to be enabled
        service.update_device(UpdateDevice( uuid= device_uuid, enabled=True))
        
        # Printing the device information after the update
        my_device = service.get_device(device_uuid).root.result[0]
        
        print(f"🆔 UUID:               {my_device.uuid}")
        print(f"📛 Name:               {my_device.name}")
        print(f"✅ Enabled:            {my_device.enabled}")

🆔 UUID:               da5d338c-8ff8-4a69-98d6-6d897cfd3bf3
📛 Name:               dummy-iosxr-demo
✅ Enabled:            True


---

## 🗑️ Method 4: Delete a Device

**Best for:** Removing decommissioned devices, cleaning up dummy entries after testing, or rotating inventory as part of an automation workflow.

**How it works:**
1. Locate the target device's UUID by iterating `list_devices()`.
2. Call `delete_device(device_uuid=...)` with the UUID.
3. Check `.root.success` on the returned `APIResult` to confirm the operation succeeded.

### 4.1 Delete a device and confirm the result

In [ ]:
from getpass import getpass
from radkit_common.utils.formatting import to_canonical_name
from radkit_service.control_api import ControlAPI

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:
        
    # Fetch the UUID of a device by its name
    device_uuid = None
    for device in service.list_devices().result:
        if device.name == to_canonical_name("dummy_iosxr_demo"):
            device_uuid = device.uuid
        
    if device_uuid is not None:
        
        # Deleting the device 
        result = service.delete_device(device_uuid=device_uuid)
        
        print(f"📱 Device {to_canonical_name('dummy_iosxr_demo')} deleted: {result.root.success}")


---

## 🔀 Which Method Should I Use?

| | 📋 Retrieve | ➕ Create | ✏️ Update | 🗑️ Delete |
|---|---|---|---|---|
| **Function** | `list_devices()` / `get_device()` | `create_device()` | `update_device()` | `delete_device()` |
| **Key input** | Device name or UUID | `NewDevice` instance | `UpdateDevice` instance | Device UUID |
| **Return type** | `StoredDevice` / list | — | — | `APIResult` |
| **Credentials needed** | Superadmin password | Superadmin password | Superadmin password | Superadmin password |
| **Best for** | Auditing, pre-flight checks | Provisioning, CI/CD | Attribute changes, toggling enabled | Decommission, cleanup |